# 05 — CLIP Embeddings
Génère les embeddings visuels (768 dims) de toutes les photos triées via `openai/clip-vit-large-patch14`.

Utilise Spark `mapInPandas` — exception justifiée aux règles no-UDF : le modèle PyTorch doit être chargé
en Python pur, il n'existe pas d'équivalent en Column expressions JVM.
Le modèle est chargé **une seule fois par partition**, pas par ligne.

**Input** : `data/raw/INSTAGRAM/CLIP_SORTING/`  
**Output** : `data/warehouse/photo_embeddings/`

In [1]:
import sys, os
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../..')))
from config import PROJECT_ROOT, WAREHOUSE

PROJECT_ROOT = Path(PROJECT_ROOT)
PHOTOS_DIR   = PROJECT_ROOT / "data/raw/INSTAGRAM/CLIP_SORTING"
OUTPUT_DIR   = Path(WAREHOUSE) / "photo_embeddings"

CLIP_MODEL   = "openai/clip-vit-large-patch14"  # 768 dims
IMAGE_EXTS   = {".jpg", ".jpeg", ".png", ".webp"}
BATCH_SIZE   = 32
N_PARTITIONS = 4

print(f"Racine projet : {PROJECT_ROOT}")
print(f"Photos dir    : {PHOTOS_DIR}")
print(f"Output dir    : {OUTPUT_DIR}")

Racine projet : /opt/spark
Photos dir    : /opt/spark/data/raw/INSTAGRAM/CLIP_SORTING
Output dir    : /opt/spark/data/warehouse/photo_embeddings


## 1. Collecter les chemins d'images

In [2]:
photo_paths = [
    {"path": str(p), "filename": p.name}
    for p in PHOTOS_DIR.rglob("*")
    if p.suffix.lower() in IMAGE_EXTS
]
print(f"Photos trouvées : {len(photo_paths)}")

Photos trouvées : 2419


## 2. Spark Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType, FloatType, StringType, StructField, StructType

spark = (
    SparkSession.builder
    .appName("MyDigitalTwin-CLIP-Embeddings")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

input_schema = StructType([
    StructField("path",     StringType(), nullable=False),
    StructField("filename", StringType(), nullable=False),
])

df = (
    spark.createDataFrame(photo_paths, schema=input_schema)
         .repartition(N_PARTITIONS)
)
print(f"DataFrame Spark : {df.count()} lignes, {N_PARTITIONS} partitions")

26/04/14 20:17:55 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


DataFrame Spark : 2419 lignes, 4 partitions


## 3. Fonction d'embedding (une fois par partition)

In [4]:
EMBED_SCHEMA = StructType([
    StructField("path",      StringType(),           nullable=False),
    StructField("filename",  StringType(),           nullable=False),
    StructField("embedding", ArrayType(FloatType()), nullable=True),
])

def embed_partition(iterator):
    import torch
    from PIL import Image
    # CLIPVisionModelWithProjection : output.image_embeds est toujours un tensor
    from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor

    device    = "cuda" if torch.cuda.is_available() else "cpu"
    processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)
    model     = CLIPVisionModelWithProjection.from_pretrained(CLIP_MODEL).to(device)
    model.eval()

    for pdf in iterator:
        paths      = pdf["path"].tolist()
        embeddings = []

        for i in range(0, len(paths), BATCH_SIZE):
            batch = paths[i : i + BATCH_SIZE]
            images, valid_idx = [], []

            for j, p in enumerate(batch):
                try:
                    images.append(Image.open(p).convert("RGB"))
                    valid_idx.append(j)
                except OSError:
                    pass

            result = [None] * len(batch)
            if images:
                inputs = processor(images=images, return_tensors="pt").to(device)
                with torch.no_grad():
                    feats = model(**inputs).image_embeds          # tensor garanti
                    feats = feats / feats.norm(dim=-1, keepdim=True)
                    feats = feats.cpu().float().numpy()
                for k, vi in enumerate(valid_idx):
                    result[vi] = feats[k].tolist()

            embeddings.extend(result)

        pdf = pdf.copy()
        pdf["embedding"] = embeddings
        yield pdf

## 4. Inférence CLIP + sauvegarde

In [5]:
df_embedded = df.mapInPandas(embed_partition, schema=EMBED_SCHEMA)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(
    df_embedded
    .filter("embedding IS NOT NULL")
    .write
    .mode("overwrite")
    .parquet(str(OUTPUT_DIR))
)

count = spark.read.parquet(str(OUTPUT_DIR)).count()
print(f"Embeddings sauvegardés : {count} photos → {OUTPUT_DIR}")

Embeddings sauvegardés : 2419 photos → /opt/spark/data/warehouse/photo_embeddings


In [6]:
spark.stop()